# Project 3: Retail Demand Forecasting - Advanced V2

This is the **V2 Advanced** notebook. In V1, out-of-the-box models severely underfit the data because real-world retail demand is highly intermittent (lots of zeros and sudden spikes).

## V2 Upgrades
1. **Steady Demand**: Upgraded from ARIMA to **SARIMAX** to capture weekly seasonal shocks (order=(5,1,2), seasonal_order=(1,1,1,7)).
2. **Seasonal Demand**: Upgraded **Prophet** to use `seasonality_mode='multiplicative'` to multiply the variance of the spikes rather than adding a flat curve.
3. **Volatile Demand**: Upgraded the **PyTorch LSTM**:
   - Increased network depth (128 hidden size, 3 layers).
   - Replaced MSE Loss with **Huber Loss** (which ignores massive outlier spikes and fits the actual curve).
   - Added an **AdamW** optimizer with a dynamic Learning Rate Scheduler.
   - **IMPORTANT: Make sure Kaggle GPU Accelerator is turned ON!**

In [ ]:
!pip install prophet
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from prophet import Prophet
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

### 1. Load the Data

In [ ]:
print("Loading M5 Dataset...")
df = pd.read_csv('/kaggle/input/m5-forecasting-accuracy/sales_train_validation.csv')

steady_item_id = 'FOODS_3_090_CA_1_validation'
seasonal_item_id = 'HOBBIES_1_234_CA_1_validation'
volatile_item_id = 'HOUSEHOLD_1_118_CA_1_validation'

steady_ts = df[df['id'] == steady_item_id].iloc[:, 6:].T
seasonal_ts = df[df['id'] == seasonal_item_id].iloc[:, 6:].T
volatile_ts = df[df['id'] == volatile_item_id].iloc[:, 6:].T

dates = pd.date_range(start='2011-01-29', periods=1913, freq='D')
steady_ts.index = dates
seasonal_ts.index = dates
volatile_ts.index = dates

steady_ts.columns = ['Sales']
seasonal_ts.columns = ['Sales']
volatile_ts.columns = ['Sales']

### 2. Advanced SARIMAX for Steady Demand

In [ ]:
train_steady = steady_ts[:-30]
test_steady = steady_ts[-30:]

print("Training Advanced SARIMAX...")
arima_model = ARIMA(train_steady, order=(5, 1, 2), seasonal_order=(1, 1, 1, 7))
arima_fit = arima_model.fit()

arima_preds = arima_fit.forecast(steps=30)

plt.figure(figsize=(10,4))
plt.plot(test_steady.index, test_steady['Sales'], label='Actual')
plt.plot(test_steady.index, arima_preds, color='red', label='SARIMAX Forecast')
plt.title('SARIMAX on Steady Demand (30-day forecast)')
plt.legend()
plt.show()

### 3. Multiplicative Prophet for Seasonal Demand

In [ ]:
prophet_df = seasonal_ts.reset_index()
prophet_df.columns = ['ds', 'y']
train_seasonal = prophet_df[:-30]
test_seasonal = prophet_df[-30:]

print("Training Advanced Multiplicative Prophet...")
m = Prophet(yearly_seasonality=True, weekly_seasonality=True, seasonality_mode='multiplicative', changepoint_prior_scale=0.05)
m.fit(train_seasonal)

future = m.make_future_dataframe(periods=30)
forecast = m.predict(future)

plt.figure(figsize=(10,4))
plt.plot(test_seasonal['ds'], test_seasonal['y'], label='Actual')
plt.plot(test_seasonal['ds'], forecast['yhat'][-30:], color='green', label='Multiplicative Prophet Forecast')
plt.title('Multiplicative Prophet on Seasonal Demand (30-day forecast)')
plt.legend()
plt.show()

### 4. Deep PyTorch LSTM for Volatile Demand

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cpu':
    print("WARNING: YOU ARE RUNNING ON CPU! TURN ON THE KAGGLE GPU ACCELERATOR FOR FASTER TRAINING!")

scaler = MinMaxScaler(feature_range=(-1, 1))
scaled_data = scaler.fit_transform(volatile_ts.values)

def create_sequences(data, seq_length):
    xs, ys = [], []
    for i in range(len(data)-seq_length-1):
        xs.append(data[i:(i+seq_length)])
        ys.append(data[i+seq_length])
    return np.array(xs), np.array(ys)

seq_length = 60 # Look back 60 days to predict the next
X, y = create_sequences(scaled_data, seq_length)

train_size = len(X) - 30
X_train = torch.FloatTensor(X[:train_size]).to(device)
y_train = torch.FloatTensor(y[:train_size]).to(device)
X_test = torch.FloatTensor(X[train_size:]).to(device)
y_test = torch.FloatTensor(y[train_size:]).to(device)

class AdvancedLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=128, num_layers=3, batch_first=True, dropout=0.2)
        self.fc1 = nn.Linear(128, 64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc1(out[:, -1, :])
        out = self.relu(out)
        out = self.fc2(out)
        return out

model = AdvancedLSTM().to(device)
criterion = nn.HuberLoss() # Huber loss ignores extreme outliers unlike MSE
optimizer = torch.optim.AdamW(model.parameters(), lr=0.005, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=10, factor=0.5)

print("Training Deep PyTorch LSTM...")
for epoch in range(200):
    model.train()
    optimizer.zero_grad()
    out = model(X_train)
    loss = criterion(out, y_train)
    loss.backward()
    optimizer.step()
    scheduler.step(loss)
    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}, LR: {optimizer.param_groups[0]['lr']:.6f}")

model.eval()
with torch.no_grad():
    preds = model(X_test).cpu().numpy()

preds_inverse = scaler.inverse_transform(preds)
actual_inverse = scaler.inverse_transform(y_test.cpu().numpy())

plt.figure(figsize=(10,4))
plt.plot(volatile_ts.index[-30:], actual_inverse, label='Actual')
plt.plot(volatile_ts.index[-30:], preds_inverse, color='orange', label='LSTM Forecast')
plt.title('Deep PyTorch LSTM on Volatile Demand (30-day forecast)')
plt.legend()
plt.show()